In [1]:
import sys
import site

print("Python executable:", sys.executable)
print("User site enabled:", site.ENABLE_USER_SITE)
print("sys.path first entries:")
for p in sys.path[:8]:
    print(p)

Python executable: /home/fjh3941/.conda/envs/cv_torch/bin/python
User site enabled: False
sys.path first entries:
/home/fjh3941/.conda/envs/cv_torch/lib/python311.zip
/home/fjh3941/.conda/envs/cv_torch/lib/python3.11
/home/fjh3941/.conda/envs/cv_torch/lib/python3.11/lib-dynload

/home/fjh3941/.conda/envs/cv_torch/lib/python3.11/site-packages


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA build version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))
    x = torch.randn(3, 3).cuda()
    print("Tensor is on:", x.device)
else:
    print("CUDA is not available in this environment/kernel.")

PyTorch version: 2.10.0
CUDA build version: 12.9
CUDA available: True
GPU count: 1
Current GPU: NVIDIA H100 80GB HBM3
Tensor is on: cuda:0


In [3]:
import os
import math
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision import transforms, datasets
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import Subset
from tqdm import tqdm
from datasets import load_from_disk


In [4]:
# -------------------------
# Config
# -------------------------

DATA_DIR = "processed_animals_256"

BATCH_SIZE = 64
EPOCHS = 300
LR = 2e-4
TIMESTEPS = 1000
DEBUG_NUM_IMAGES = 100
IMG_SIZE = 256
# BATCH_SIZE = 8
# EPOCHS = 50
# LR = 2e-4
# TIMESTEPS = 200
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("samples", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)


# -------------------------
# Dataset
# -------------------------

from tqdm import tqdm

print("Loading from disk...")
raw = load_from_disk(DATA_DIR)
print(f"Dataset size: {len(raw)}")

print("Converting to numpy in chunks...")
raw.set_format(None)
chunk_size = 500
chunks = []

for start in tqdm(range(0, len(raw), chunk_size), desc="Loading"):
    end   = min(start + chunk_size, len(raw))
    batch = raw[start:end]["pixel_values"]
    chunks.append(np.array(batch))

all_images = np.concatenate(chunks, axis=0)
print(f"Loaded shape: {all_images.shape}")

all_images = torch.tensor(all_images, dtype=torch.float32).permute(0, 3, 1, 2) / 127.5 - 1.0

print(f"Moving to GPU... ({all_images.nbytes / 1e9:.2f} GB)")
all_images = all_images.to(DEVICE)
print("Done.")

class CachedDataset(Dataset):
    def __init__(self, images):
        self.images = images
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.images[idx]

dataset = CachedDataset(all_images)

loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=0, pin_memory=False)

Loading from disk...
Dataset size: 5400
Converting to numpy in chunks...


Loading: 100%|██████████| 11/11 [13:12<00:00, 72.03s/it]


Loaded shape: (5400, 256, 256, 3)
Moving to GPU... (4.25 GB)
Done.


In [12]:

# -------------------------
# Offset Cosine Diffusion Schedule
# -------------------------

def offset_cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)

    alpha_hats = torch.cos(
        ((x / timesteps) + s) / (1 + s) * math.pi * 0.5
    ) ** 2

    alpha_hats = alpha_hats / alpha_hats[0]

    betas = 1 - (alpha_hats[1:] / alpha_hats[:-1])
    betas = torch.clip(betas, 1e-4, 0.999)

    return betas

betas      = offset_cosine_beta_schedule(TIMESTEPS).to(DEVICE)
alphas     = (1.0 - betas).to(DEVICE)
alpha_hats = torch.cumprod(alphas, dim=0).to(DEVICE)


def add_noise(x, t):
    noise = torch.randn_like(x)

    alpha_hat = alpha_hats[t].view(-1, 1, 1, 1)

    noisy_x = torch.sqrt(alpha_hat) * x + torch.sqrt(1 - alpha_hat) * noise

    return noisy_x, noise


# -------------------------
# Time Embedding
# -------------------------

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb_scale = math.log(10000) / (half_dim - 1)

        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb_scale)
        emb = t[:, None] * emb[None, :]

        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)


# -------------------------
# U-Net Blocks
# -------------------------

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.norm1 = nn.GroupNorm(8, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)

        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        time_emb = self.time_mlp(t)
        h = h + time_emb[:, :, None, None]

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + self.skip(x)


class UNet(nn.Module):
    def __init__(self, img_channels=3, base_channels=64, time_dim=256):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.enc1 = ResBlock(img_channels, base_channels, time_dim)
        self.enc2 = ResBlock(base_channels, base_channels * 2, time_dim)
        self.enc3 = ResBlock(base_channels * 2, base_channels * 4, time_dim)
        self.enc4 = ResBlock(base_channels * 4, base_channels * 8, time_dim)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ResBlock(base_channels * 8, base_channels * 8, time_dim)

        # Decoder
        self.up4 = nn.ConvTranspose2d(512, 512, kernel_size=2, stride=2)
        self.dec4 = ResBlock(1024, 256, time_dim)  # 512 + 512

        self.up3 = nn.ConvTranspose2d(256, 256, kernel_size=2, stride=2)
        self.dec3 = ResBlock(512, 128, time_dim)   # 256 + 256

        self.up2 = nn.ConvTranspose2d(128, 128, kernel_size=2, stride=2)
        self.dec2 = ResBlock(256, 64, time_dim)    # 128 + 128

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = ResBlock(128, 64, time_dim)    # 64 + 64

        self.out = nn.Conv2d(64, 3, kernel_size=1)

    def forward(self, x, t):
        t = self.time_embedding(t)

        e1 = self.enc1(x, t)
        e2 = self.enc2(self.pool(e1), t)
        e3 = self.enc3(self.pool(e2), t)
        e4 = self.enc4(self.pool(e3), t)

        b = self.bottleneck(self.pool(e4), t)

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4, t)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3, t)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2, t)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1, t)

        return self.out(d1)


# -------------------------
# Sampling
# -------------------------

@torch.no_grad()
def sample(model, n=4):
    model.eval()

    x = torch.randn(n, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

    for i in tqdm(reversed(range(TIMESTEPS)), desc="Sampling"):
        t = torch.full((n,), i, device=DEVICE, dtype=torch.long)

        predicted_noise = model(x, t)

        alpha = alphas[t].view(-1, 1, 1, 1)
        alpha_hat = alpha_hats[t].view(-1, 1, 1, 1)
        beta = betas[t].view(-1, 1, 1, 1)

        if i > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = (1 / torch.sqrt(alpha)) * (
            x - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * predicted_noise
        ) + torch.sqrt(beta) * noise

    x = torch.clamp(x, -1, 1)
    x = (x + 1) / 2

    return x


In [13]:
print(betas.device)       # should print cuda:0
print(alphas.device)      # should print cuda:0
print(alpha_hats.device)  # should print cuda:0

cuda:0
cuda:0
cuda:0


Training

In [14]:
import time
import glob
import torch.backends.cudnn as cudnn
from torchvision.utils import save_image
from tqdm import tqdm

# ── Setup ────────────────────────────────────────────────
cudnn.benchmark = True

os.makedirs("samples",     exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

LR = 5e-5

model     = UNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler    = torch.amp.GradScaler('cuda')

loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=0, pin_memory=False)

# ── Training loop ────────────────────────────────────────
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    start      = time.time()

    loop = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images in loop:
        t = torch.randint(0, TIMESTEPS, (images.size(0),), device=DEVICE).long()

        with torch.amp.autocast('cuda'):
            noisy_images, noise = add_noise(images, t)
            predicted_noise     = model(noisy_images, t)
            loss                = F.mse_loss(predicted_noise, noise)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss   = total_loss / len(loader)
    elapsed    = time.time() - start
    mins, secs = divmod(int(elapsed), 60)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Time: {mins}m {secs}s")
    
    if (epoch + 1) % 10 == 0:
        # ── Sample & save comparison image ──────────────────────────────
        model.eval()
        with torch.no_grad():
            # grab 4 real images from the dataset
            real_images = all_images[:4]  

            # add noise at a mid timestep then denoise back
            t_sample = torch.full((4,), 500, device=DEVICE, dtype=torch.long)
            noisy, _ = add_noise(real_images, t_sample)

            # denoise from that timestep back to 0
            x = noisy
            for i in tqdm(reversed(range(500)), desc="Sampling", leave=False):
                t_s = torch.full((4,), i, device=DEVICE, dtype=torch.long)

                with torch.amp.autocast('cuda'):
                    predicted_noise = model(x, t_s)

                alpha     = alphas[t_s].view(-1, 1, 1, 1)
                alpha_hat = alpha_hats[t_s].view(-1, 1, 1, 1)
                beta      = betas[t_s].view(-1, 1, 1, 1)
                noise     = torch.randn_like(x) if i > 0 else torch.zeros_like(x)

                x = (1 / torch.sqrt(alpha)) * (
                    x - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * predicted_noise
                ) + torch.sqrt(beta) * noise

            # normalise all three sets to [0, 1] for saving
            real_grid   = (torch.clamp(real_images, -1, 1) + 1) / 2
            noisy_grid  = (torch.clamp(noisy,       -1, 1) + 1) / 2
            recon_grid  = (torch.clamp(x,           -1, 1) + 1) / 2

            # stack as: real | noisy | reconstructed
            comparison = torch.cat([real_grid, noisy_grid, recon_grid], dim=0)
            save_image(comparison, f"samples/epoch_{epoch+1:04d}.png", nrow=4)

    model.train()

    # ── Save checkpoint ──────────────────────────────────
    torch.save({
        "epoch":           epoch + 1,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scaler_state":    scaler.state_dict(),
        "loss":            avg_loss,
    }, f"checkpoints/epoch_{epoch+1:04d}.pt")

    # Keep only the last 5 checkpoints
    for old in sorted(glob.glob("checkpoints/epoch_*.pt"))[:-5]:
        os.remove(old)

torch.save(model.state_dict(), "final_model.pt")
print("Final model saved.")

Epoch 1/300: 100%|██████████| 85/85 [00:14<00:00,  5.81it/s, loss=0.1]  


Epoch 1/300 | Loss: 0.3926 | Time: 0m 14s


Epoch 2/300: 100%|██████████| 85/85 [00:14<00:00,  6.07it/s, loss=0.0858]


Epoch 2/300 | Loss: 0.0940 | Time: 0m 14s


Epoch 3/300: 100%|██████████| 85/85 [00:14<00:00,  6.03it/s, loss=0.0758]


...

Epoch 299/300: 100%|██████████| 85/85 [00:14<00:00,  6.05it/s, loss=0.0255]


Epoch 299/300 | Loss: 0.0268 | Time: 0m 14s


Epoch 300/300: 100%|██████████| 85/85 [00:13<00:00,  6.07it/s, loss=0.0418]


Epoch 300/300 | Loss: 0.0273 | Time: 0m 13s


Final model saved.
